# L'emprise d'évaluation change le classement

Ce carnet rejoue la section 6, qui est le résultat de méthode de
l'article : **une AUC n'a de sens qu'accompagnée de la population sur laquelle
elle a été calculée.**

Deux tables :

- `exp43_fusion_ciblee_coherence.csv` — les AUC par produit sur l'emprise
  commune aux produits à large couverture, 148 159 bâtiments ;
- `exp42_aoi_commun_face_a_face.csv` — le face-à-face, produit par produit, sur
  les seuls bâtiments que les deux couvrent.

In [ ]:
import os
import pandas as pd

# les carnets se lancent depuis notebooks/ ; les tables sont dans data/results
RESULTATS = os.path.join("..", "data", "results")
TABLES = os.path.join("..", "tables")

def lire(dossier, nom, **kw):
    """Lit une table du depot. Le separateur et la virgule decimale varient
    d'une campagne a l'autre : on laisse pandas les deviner sauf indication."""
    kw.setdefault("sep", None)
    kw.setdefault("engine", "python")
    return pd.read_csv(os.path.join(dossier, nom), **kw)

def presque(a, b, tol=0.0006):
    """Egalite au millieme pres, tolerance par defaut d'un demi-millieme."""
    assert abs(a - b) < tol, "%.4f attendu, %.4f obtenu" % (b, a)
    return True


Le classement sur l'emprise strictement commune.

In [ ]:
e = lire(RESULTATS, "exp43_fusion_ciblee_coherence.csv")
commun = e[e.lecture == "A - emprise des huit"].copy()
seuls = commun[commun.score.str.endswith("seul")].sort_values("moyenne",
                                                             ascending=False)
print(seuls[["score", "auc_ems", "auc_chatmap", "moyenne"]]
      .to_string(index=False))
print("\nbatiments de l'emprise commune :", int(commun.n_emprise.iloc[0]))

Les trois chiffres que la page d'accueil met en avant.

In [ ]:
# les chiffres de tete de l'article
tstat = seuls.loc[seuls.score == "notre T-stat seul", "moyenne"].iloc[0]
osu = seuls.loc[seuls.score == "OSU seul", "moyenne"].iloc[0]
presque(tstat, 0.7403)
presque(osu, 0.7324)
assert int(commun.n_emprise.iloc[0]) == 148159
print("Verifie : T-stat 0,7403 ; OSU 0,7324 ; 148 159 batiments.")

Le face-à-face, où les écarts sont les plus larges — et les échantillons les plus inégaux.

In [ ]:
ff = lire(RESULTATS, "exp42_aoi_commun_face_a_face.csv")
ff = ff.sort_values("ecart_chatmap", ascending=False)
print(ff[["adversaire", "n_commun", "auc_tstat_ems", "auc_adversaire_ems",
          "auc_tstat_chatmap", "auc_adversaire_chatmap"]].to_string(index=False))

Pourquoi un face-à-face ne se résume pas en un seul nombre.

In [ ]:
# la comparaison en face-a-face n'est equitable que paire par paire : les
# effectifs varient de deux ordres de grandeur.
print("batiments communs : de %d a %d selon l'adversaire"
      % (ff.n_commun.min(), ff.n_commun.max()))

La troisieme lecture : chaque produit note sur **son propre perimetre**, celui
des batiments qu'il couvre effectivement. C'est la serie grise de la figure 11,
et c'est elle qui explique pourquoi le classement change — OSU passe de 0,802
sur son emprise a 0,732 sur l'emprise commune, tandis que notre T-stat ne bouge
presque pas.

In [ ]:
propre = lire(TABLES, "product_ranking_own_footprint.csv")
print(propre[["Rang", "Produit", "AUC / Copernicus", "AUC / ChatMap",
              "Moyenne"]].to_string(index=False))

# La table de synthese doit redonner exactement l'experience 44, qui en est
# la source : moyenne des deux verites, produit par produit.
q = lire(RESULTATS, "exp44_six_questions.csv")
q = q[q.mesure.str.startswith("emprise propre")]
source = {}
for _, r in q.iterrows():
    nom, quoi = r.mesure.split(" — ", 1)[1].rsplit(", ", 1)
    source.setdefault(nom, {})[quoi] = float(r.valeur)

def virgule(x):
    """Les tables de synthese sont ecrites a la francaise."""
    return float(str(x).replace(",", "."))

for _, r in propre.iterrows():
    d = source[r.Produit]
    presque(virgule(r["AUC / Copernicus"]), d["vs Copernicus"])
    presque(virgule(r["AUC / ChatMap"]), d["vs ChatMap"])
    presque(virgule(r["Moyenne"]), d["moyenne des deux verites"])
print()
print("Verifie : les douze lignes de la table viennent de l'experience 44.")

Les trois chiffres de la serie grise, et ce qu'ils font au classement.

In [ ]:
# Ce que montre la figure 11 : l'emprise propre ne classe pas comme l'emprise
# commune, et seul le T-stat garde la meme valeur dans les deux lectures.
prop = {r.Produit: virgule(r["Moyenne"]) for _, r in propre.iterrows()}
comm = {r.score[:-5]: r.moyenne for _, r in seuls.iterrows()}

presque(prop["OSU"], 0.8019)
presque(prop["notre T-stat"], 0.7487)
presque(prop["NASA DRCS S2"], 0.7486)
assert propre.iloc[0].Produit == "OSU"

ecart_tstat = abs(prop["notre T-stat"] - comm["notre T-stat"])
ecart_osu = abs(prop["OSU"] - comm["OSU"])
print("ecart entre les deux lectures : T-stat %.4f, OSU %.4f"
      % (ecart_tstat, ecart_osu))
assert ecart_tstat < 0.01 < ecart_osu
print()
print("Verifie : OSU est premier sur son emprise propre (0,8019) et le")
print("T-stat deuxieme (0,7487). Sur l'emprise commune l'ordre s'inverse,")
print("parce qu'OSU perd 0,07 en changeant de population la ou le T-stat")
print("n'en perd pas 0,01.")